# Random Forest Regression HW

## 1. Linear regression (со старого дз)

In [1636]:
import pandas as pd
from sklearn.model_selection import train_test_split,  GridSearchCV
from sklearn.preprocessing import StandardScaler , MinMaxScaler , OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

df = pd.read_csv('ToyotaCorolla.csv')

## Датасет:

In [1637]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 39 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Id                 1436 non-null   int64
 1   Model              1436 non-null   str  
 2   Price              1436 non-null   int64
 3   Age_08_04          1436 non-null   int64
 4   Mfg_Month          1436 non-null   int64
 5   Mfg_Year           1436 non-null   int64
 6   KM                 1436 non-null   int64
 7   Fuel_Type          1436 non-null   str  
 8   HP                 1436 non-null   int64
 9   Met_Color          1436 non-null   int64
 10  Color              1436 non-null   str  
 11  Automatic          1436 non-null   int64
 12  CC                 1436 non-null   int64
 13  Doors              1436 non-null   int64
 14  Cylinders          1436 non-null   int64
 15  Gears              1436 non-null   int64
 16  Quarterly_Tax      1436 non-null   int64
 17  Weight             1436 n

In [1638]:
df.describe()

,Id,Price,Age_08_04,Mfg_Month,Mfg_Year,KM,HP,Met_Color,Automatic,CC,...,Powered_Windows,Power_Steering,Radio,Mistlamps,Sport_Model,Backseat_Divider,Metallic_Rim,Radio_cassette,Parking_Assistant,Tow_Bar
count,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.00000,...,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000
mean,721.555014,10730.824513,55.947075,5.548747,1999.625348,68533.259749,101.502089,0.674791,0.055710,1576.85585,...,0.561978,0.977716,0.146240,0.256964,0.300139,0.770195,0.204735,0.145543,0.002786,0.277855
std,416.476890,3626.964585,18.599988,3.354085,1.540722,37506.448872,14.981080,0.468616,0.229441,424.38677,...,0.496317,0.147657,0.353469,0.437111,0.458478,0.420854,0.403649,0.352770,0.052723,0.448098
min,1.000000,4350.000000,1.000000,1.000000,1998.000000,1.000000,69.000000,0.000000,0.000000,1300.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,361.750000,8450.000000,44.000000,3.000000,1998.000000,43000.000000,90.000000,0.000000,0.000000,1400.00000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,721.500000,9900.000000,61.000000,5.000000,1999.000000,63389.500000,110.000000,1.000000,0.000000,1600.00000,...,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,1081.250000,11950.000000,70.000000,8.000000,2001.000000,87020.750000,110.000000,1.000000,0.000000,1600.00000,...,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1442.000000,32500.000000,80.000000,12.000000,2004.000000,243000.000000,192.000000,1.000000,1.000000,16000.00000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [1639]:
df.describe(include='str')

,Model,Fuel_Type,Color
count,1436,1436,1436
unique,319,3,10
top,TOYOTA Corolla 1.6 16V HATCHB LINEA TERRA 2/3-...,Petrol,Grey
freq,109,1264,301


In [1640]:
NaN = df.isnull().sum().value_counts()
print(NaN)

0    39
Name: count, dtype: int64


## Предобработка: удаление скоррелированых/не нужных для поставленной задачи данных

In [1641]:
# корелляция между циллиндрами и объемом

print(df['Cylinders'].corr(df['CC']))

nan


/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


1) Cylinders содержит единственное значение 4, значит удаляем

In [1642]:
# корелляция между передачами и объемом

print(df['Gears'].corr(df['CC']))

0.014629352089597158


### Итак, ненужными/бесполезными я счел следующие признаки: 
 - Cylinders - единственное значение
 - Weight - несущественный показатель
 - Backseat_Divider - не существенный показатель
 - Tow_Bar - несущественный показатель
 - Radio_cassette - несущественный (устаревший) показатель
 - CD_Player - несущественный (устаревший) показатель
 - Met_Color - несущественный (устаревший) показатель
 - Metallic_Rim - несущественный (устаревший) показатель
 - Mistlamps - несущественный (устаревший) показатель
 - Model - содержит слишком много значений

In [1643]:

cols_to_drop = [
    'Id',
    'Cylinders',
    'Weight',
    'Backseat_Divider',
    'Tow_Bar',
    'Radio_cassette',
    'CD_Player',
    'Met_Color',
    'Metallic_Rim',
    'Mistlamps',
    'Model'
]

df = df.drop(columns=cols_to_drop)

## Feature Ingeneering

### Создам 1 колонку с возрастом вместо 3 разных

In [1644]:
base_year = 2004
base_month = 8
df['Age_Month'] = (base_year*12 + base_month) - df['Mfg_Year']*12 + df['Mfg_Month'] + df['Age_08_04']

df = df.drop(columns=['Mfg_Year', 'Mfg_Month', 'Age_08_04'])

In [1645]:
print(df['Age_Month'].corr(df['KM']))

0.5049744502001694


### One-hot encoding для оставшихся str признаков

In [1646]:
df = pd.get_dummies(df, columns=['Fuel_Type'], prefix='Fuel', drop_first=True)

In [1647]:
print(df['Color'].value_counts())

Color
Grey      301
Blue      283
Red       278
Green     220
Black     191
Silver    122
White      31
Violet      4
Yellow      3
Beige       3
Name: count, dtype: int64


In [1648]:
df['Color'] = df['Color'].apply(lambda x: 'Other' if x in ['Violet', 'Yellow', 'Beige'] else x)

df = pd.get_dummies(df, columns=['Color'], prefix='Color')
df = df.drop(columns='Color_Other')

In [1649]:
df.info()
df.describe(include=['int64', 'bool'])

<class 'pandas.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Price              1436 non-null   int64
 1   KM                 1436 non-null   int64
 2   HP                 1436 non-null   int64
 3   Automatic          1436 non-null   int64
 4   CC                 1436 non-null   int64
 5   Doors              1436 non-null   int64
 6   Gears              1436 non-null   int64
 7   Quarterly_Tax      1436 non-null   int64
 8   Mfr_Guarantee      1436 non-null   int64
 9   BOVAG_Guarantee    1436 non-null   int64
 10  Guarantee_Period   1436 non-null   int64
 11  ABS                1436 non-null   int64
 12  Airbag_1           1436 non-null   int64
 13  Airbag_2           1436 non-null   int64
 14  Airco              1436 non-null   int64
 15  Automatic_airco    1436 non-null   int64
 16  Boardcomputer      1436 non-null   int64
 17  Central_Lock       1436 n

,Price,KM,HP,Automatic,CC,Doors,Gears,Quarterly_Tax,Mfr_Guarantee,BOVAG_Guarantee,...,Age_Month,Fuel_Diesel,Fuel_Petrol,Color_Black,Color_Blue,Color_Green,Color_Grey,Color_Red,Color_Silver,Color_White
count,1436.000000,1436.000000,1436.000000,1436.000000,1436.00000,1436.000000,1436.000000,1436.000000,1436.000000,1436.000000,...,1436.000000,1436,1436,1436,1436,1436,1436,1436,1436,1436
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2,2,2,2,2,2,2,2,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,False,True,False,False,False,False,False,False,False
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1281,1264,1245,1153,1216,1135,1158,1314,1405
mean,10730.824513,68533.259749,101.502089,0.055710,1576.85585,4.033426,5.026462,87.122563,0.409471,0.895543,...,121.991643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,3626.964585,37506.448872,14.981080,0.229441,424.38677,0.952677,0.188510,41.128611,0.491907,0.305959,...,36.977325,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,4350.000000,1.000000,69.000000,0.000000,1300.00000,2.000000,3.000000,19.000000,0.000000,0.000000,...,17.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,8450.000000,43000.000000,90.000000,0.000000,1400.00000,3.000000,5.000000,69.000000,0.000000,1.000000,...,89.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,9900.000000,63389.500000,110.000000,0.000000,1600.00000,4.000000,5.000000,85.000000,0.000000,1.000000,...,137.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,11950.000000,87020.750000,110.000000,0.000000,1600.00000,5.000000,5.000000,85.000000,1.000000,1.000000,...,161.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Разделение выборки данных

In [1650]:
df_X = df.drop(columns='Price')
df_Y = df['Price']

x_train, x_test, y_train, y_test = train_test_split(df_X, df_Y, test_size=0.3, random_state=42)

### Я решил разделить выборку в соотношении 70/30, ибо данных для обучения достаточно, и есть смысл оставить побольше для проверки.

### разделение данных на такие выборки нужно, ибо модель так или иначе подстраивается под данные, на которых она учится, поэтому проверка ее качества на данных, которых она училась необъективна и может привести к переобучению

## Обучение модели

In [1651]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [1652]:
model = LinearRegression()

model.fit(x_train, y_train)

y_pred_train = model.predict(x_train)
y_pred = model.predict(x_test)


mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

mse_test = mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
mape_test = mean_absolute_percentage_error(y_test, y_pred)

print(f'MSE train: {mse_train}')
print(f'R^2 train: {r2_train}')
print(f'MAPE train: {mape_train}')

print(f'MSE test: {mse_test}')
print(f'R^2 test: {r2_test}')
print(f'MAPE test: {mape_test}')

MSE train: 1258028.5982832033
R^2 train: 0.9018141506365719
MAPE train: 0.08318474500759947
MSE test: 1377976.5441248151
R^2 test: 0.9010226571818616
MAPE test: 0.08651520197823764


#### вроде нормально и так, но я попробую, что будет для регуляризации

In [1653]:
ridge = Ridge()

ridge.fit(x_train, y_train)

y_pred_train = ridge.predict(x_train)
y_pred = ridge.predict(x_test)


mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

mse_test = mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
mape_test = mean_absolute_percentage_error(y_test, y_pred)

print(f'MSE train: {mse_train}')
print(f'R^2 train: {r2_train}')
print(f'MAPE train: {mape_train}')

print(f'MSE test: {mse_test}')
print(f'R^2 test: {r2_test}')
print(f'MAPE test: {mape_test}')

MSE train: 1258070.6081764337
R^2 train: 0.901810871874027
MAPE train: 0.0831737765424839
MSE test: 1377807.2500584512
R^2 test: 0.9010348172414178
MAPE test: 0.08648361186580503


In [1654]:
lasso = Lasso()

lasso.fit(x_train, y_train)

y_pred_train = lasso.predict(x_train)
y_pred = lasso.predict(x_test)


mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

mse_test = mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
mape_test = mean_absolute_percentage_error(y_test, y_pred)

print(f'MSE train: {mse_train}')
print(f'R^2 train: {r2_train}')
print(f'MAPE train: {mape_train}')

print(f'MSE test: {mse_test}')
print(f'R^2 test: {r2_test}')
print(f'MAPE test: {mape_test}')

MSE train: 1258204.2843999977
R^2 train: 0.9018004387935961
MAPE train: 0.08316769964204868
MSE test: 1380121.7614214872
R^2 test: 0.9008685704459898
MAPE test: 0.08655258288943998


In [1655]:
# перед этим я дропну еще признаков, ибо PolynomialFeatures даже для deg = 2 добавит еще С(34, 2) = 34*33/(2!) признаков
# spoiler: это ухудшило качество, поэтому я попробую приемнить для текущего датасета

steps = [
    ('poly', PolynomialFeatures(degree=2)),
    ('model', Lasso(max_iter=100))
]
pipe = Pipeline(steps)

# + небольшой поиск руками
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 30.0, 100.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2')
grid.fit(x_train, y_train)

best = grid.best_estimator_
print(f"Best param: {grid.best_params_['model__alpha']}")

y_pred_lasso_train = best.predict(x_train)
y_pred_lasso = best.predict(x_test)

r2_train = r2_score(y_train, y_pred_lasso_train)
mape_lasso_train = mean_absolute_percentage_error(y_train, y_pred_lasso_train)
r2 = r2_score(y_test, y_pred_lasso)
mape_lasso = mean_absolute_percentage_error(y_test, y_pred_lasso)

print(f"R^2 train: {r2_train}")
print(f"R^2 test: {r2}")
print(f"MAPE train: {mape_lasso_train}")
print(f"MAPE test: {mape_lasso}")

/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.706e+08, tolerance: 1.083e+06
  model = cd_fast.enet_coordinate_descent(
/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.571e+08, tolerance: 9.757e+05
  model = cd_fast.enet_coordinate_descent(
/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t

Best param: 30.0
R^2 train: 0.9377667177073048
R^2 test: 0.9224701215179658
MAPE train: 0.068026349755823
MAPE test: 0.08104802355061057


/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.156e+06, tolerance: 1.059e+06
  model = cd_fast.enet_coordinate_descent(
/Users/alex/Desktop/ml-course-homeworks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.605e+06, tolerance: 1.288e+06
  model = cd_fast.enet_coordinate_descent(


In [1656]:
# и для ridge

steps = [
    ('poly', PolynomialFeatures(degree=2)),
    ('model', Ridge(max_iter=100))
]
pipe = Pipeline(steps)

# + небольшой поиск руками
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 30.0, 100.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2')
grid.fit(x_train, y_train)

best = grid.best_estimator_
print(f"Best param: {grid.best_params_['model__alpha']}")

y_pred_lasso_train = best.predict(x_train)
y_pred_lasso = best.predict(x_test)

r2_train = r2_score(y_train, y_pred_lasso_train)
mape_lasso_train = mean_absolute_percentage_error(y_train, y_pred_lasso_train)
r2 = r2_score(y_test, y_pred_lasso)
mape_lasso = mean_absolute_percentage_error(y_test, y_pred_lasso)

print(f"R^2 train: {r2_train}")
print(f"R^2 test: {r2}")
print(f"MAPE train: {mape_lasso_train}")
print(f"MAPE test: {mape_lasso}")

Best param: 30.0
R^2 train: 0.9584919255084561
R^2 test: 0.8778909696372038
MAPE train: 0.05483170027012634
MAPE test: 0.09579712194310143


- **Данных немного, поэтому и скорость обучения высокая**
- **Регуляризация не улучшила точность -> мультиколлинеарность отсутствует**
- **Регуляризация с полиномиальными признаками и гипепараметром для Лассо немного улучшила точность -> существует некоторая нелинейная зависимость между данными**
- **В случае Лассо с гиперпараметром предсказания для train и test примерно одинаковой точности -> модель не запомнила много шума**
- **В случае Ridge с гиперпараметром предсказания для train и test разной точность (по r2: 0.96 и 0.87 соответсвенно) -> существуют лишние/коллинеарные признаки и произошло переобучение**

## Итого

1) Использованные метрики:
    - **MSE** - для оценки суммы обьективных отклонений (не MAE, т.к. он менее чувствителен к выбросам, которые критичны для данного датасета)
    - **MAPE** - для оценки ошибки в процентах (усредненной)
    - **R^2** - для оценки дисперсии нашей цены исходя из работы модели

2) На какой выборке метрики:
    - Также я считал метрики для тестовой и тренировочной выборок, чтобы оценить влияние шума и риски переобучения

3) Лучше по итогу справилась:
    - модель Лассо с гиперпараметром l = 30 для выборки с дополнительными полиномиальными признакакми - R^2 test: 0.92, MAPE test: 0.08. Почему лассо а не ridge - как видно из ridge, модель переобучилась - значит существуют лишние признаки, которые как раз были аннулированы регуляризацией Лассо, обнуляющей лишние признаки
    - Но также неплохо справилась модель без регуляризации и полиномиальных признаков - R^2: 0.905, MAPE: 0.085 - что говорит об отсутсвии коллизий в выборке данных

4) Насколько хорошие получились результаты:
    - В целом, меня они устроили, ибо точность довольно приемлимая (для средней цены по датасету 10730.8 модель предсказывает с точностью до 8-10% ~ +- 1000 евро (вроде евро)), однако единственное, что мне не особо нравится - большое число признаков, из-за чего регуляризация с полиномиальными признаками в случае с бОльшим числом данных сильно замедлится. С другой стороны, и без нее результат довольно точный, поэтому я в целом доволен этой работой

5) Чем докажете, что ваша модель не переобучилась?
    - Во всех случаях, кроме последнего (ridge с гиперпараметром) точность предсказантя модели почти не отличается на тренировочной и тестовой выборке, что говорит о том, что она не запомнила шум и не выучила тренировочный датасат, т.е. не переобучилась.

## 2. Random Forest Regression

### Подготовка данные для обучения модели. 

1. Предобработка будет отличаться тем, что:
- уберу StandardScaler, т.к. деревья не зависят от метрики, а только от порога
- осталвю много признаков, т.к. благодаря этому на этапе построения деревьев каждое дерево будет иметь больше уникальных признаков => скореллированность моделей ниже => дисперсия меньше. А также по этой приниче будет применим бэггинг

  (и тут оговорка - количество признаков я не буду менять по сравнению с linear regression, ибо я и так там ранее оставил довольно много)
<br>
<br>
2. Разделение выборки:
- Решил разделить выборку в соотношении 70/30 используя train_test_split
<br>
<br>
3. На сколько частей нужно делить выборку при использовании кросс-валидации?
- нужно делить на 5 - 10 частей. Если разделить на большее, то оценка качества может стать неточной
<br>
<br>
4. Можно ли не использовать кросс-валидацию? Если да, то как делить выборку в таком случае?
- можно: тогда делим выборку на обучающую, валидационную и тестовую.

In [1657]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import numpy as np

x_train_rf, x_test_rf, y_train_rf, y_test_rf = train_test_split(df_X, df_Y, test_size=0.3, random_state=42)

In [1658]:
max_depth=8
min_samples_split=2

# для регрессии - max_features = [m/3] = [33/3] = 11
rf_model = RandomForestRegressor(n_estimators=10, 
                           criterion='squared_error', 
                           max_depth=max_depth, 
                           min_samples_split=min_samples_split,
                           max_features=11,
                           bootstrap=True, 
                           n_jobs=2, 
                           random_state=42)

rf_model.fit(x_train_rf, y_train_rf)

tr_model = DecisionTreeRegressor(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
tr_model.fit(x_train_rf, y_train_rf)



rf_pred_train = rf_model.predict(x_train_rf)
tr_pred_train = tr_model.predict(x_train_rf)

rf_pred = rf_model.predict(x_test_rf)
tr_pred = tr_model.predict(x_test_rf)

In [1659]:
print(f"Accuracy (random forest): {rf_model.score(x_test_rf, y_test_rf):.2f}")
print(f"Accuracy (tree): {tr_model.score(x_test_rf, y_test_rf):.2f}")

Accuracy (random forest): 0.92
Accuracy (tree): 0.89


In [1660]:
r2_train_rf = r2_score(y_train_rf, rf_pred_train)
mape_lasso_train_rf = mean_absolute_percentage_error(y_train_rf, rf_pred_train)
r2_rf = r2_score(y_test_rf, rf_pred)
mape_lasso_rf = mean_absolute_percentage_error(y_test_rf, rf_pred)

r2_train_tr = r2_score(y_train_rf, tr_pred_train)
mape_lasso_train_tr = mean_absolute_percentage_error(y_train_rf, tr_pred_train)
r2_tr = r2_score(y_test_rf, tr_pred)
mape_lasso_tr = mean_absolute_percentage_error(y_test_rf, tr_pred)


print(f"random forest:")
print(f"    R^2 train: {r2_train_rf}")
print(f"    R^2 test: {r2_rf}")
print(f"    MAPE train: {mape_lasso_train_rf}")
print(f"    MAPE test: {mape_lasso_rf}", end='\n\n\n')
print(f"tree:")
print(f"    R^2 train: {r2_train_tr}")
print(f"    R^2 test: {r2_tr}")
print(f"    MAPE train: {mape_lasso_train_tr}")
print(f"    MAPE test: {mape_lasso_tr}", end='\n\n\n')
print(f"linear regression:")
print(f"    R^2 train: {r2_train}")
print(f"    R^2 test: {r2}")
print(f"    MAPE train: {mape_lasso_train}")
print(f"    MAPE test: {mape_lasso}")

random forest:
    R^2 train: 0.9485869748105668
    R^2 test: 0.918532078646299
    MAPE train: 0.0637671351757339
    MAPE test: 0.08353149665325701


tree:
    R^2 train: 0.9566093425921574
    R^2 test: 0.8873043953617354
    MAPE train: 0.057640002033213275
    MAPE test: 0.09144975653413136


linear regression:
    R^2 train: 0.9584919255084561
    R^2 test: 0.8778909696372038
    MAPE train: 0.05483170027012634
    MAPE test: 0.09579712194310143


1. **Сравнение скорости:** сравните, какая модель обучалась быстрее:
- В связи с тем, что датасет небольшой и довольно простой, получилось так, что обе модели обучились одинаково бысто - 0.0s. Но в предположении большего датасета, получилось бы так, что модель linreg обучилась бы быстрее, ибо, если даже на этом сете взять парамерт n_estimatiors - количество деревьев в ансамбле - 1000+, то получим, что время уже разнится, что связано с тем, что для обучения ансамбля нужно обучить каждую модель из n_estimators по отдельности.
<br>
<br>
2. **Можно ли добиться одинаковой или близкой к одинаковой скорости?**:
- Да, это можно сделать, сделав процесс обучения деревьев параллельным, ибо они обучаются независимо друг от друга. Для этого достаточно поменять параметр n_jobs в модели RandomForestRegressor.
<br>
<br>
3. **Сравнение качества:** сравните результаты одного дерева и случайного леса:
- Ожидаемо, результаты случайного леса лучше результатов 1 дерева с такими же параметрами (0.92 против 0.89 для r2_score). Это связано с самой концепцией ансамблевой модели, ибо она усредняет сумму результатов каждого из независимых деревьев (которые являются таковыми из-за бутстрапа), из которых состоит, из-за чего получается "независимый" от ошибок конкретного дерева леса результат.

**результаты модели линейной регрессии оказались чуть хуже результатов случайного леса (0.88 против 0.92 для r2_score)**